# Dvouvrstvý perceptron pro obrázků CIFAR10

- Úkolem cvičení je naprogramovat **dvouvrstvý perceptron** pro klasifikaci obrázků na datasetu CIFAR-10.
- Jediný rozdíl oproti minulému cvičení, kde jsme trénovali lineární model, je tedy použitý klasifikátor.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append("..")  # import tests, ans

import matplotlib.pyplot as plt
import numpy as np
import PIL
import torch
import torchvision

import ans
from tests import test_two_layer_perceptron

# Data

## Preprocessing

- Preprocessing dat provedeme shodně s úlohou [`linear_classification`](linear_classification.ipynb).

### TODO: implementuje funkci `preprocess`.

In [3]:
def preprocess(img: PIL.Image) -> torch.Tensor:
    """
    Args:
        img: image of type PIL.Image
    Returns:
        x: 1-dimensional tensor; shape (num_pixels * num_channels,), dtype float32
    """

    shape = np.array(img)
    x = torch.tensor(shape, dtype = torch.float32) / 255.0
    x = x.reshape(-1)

    # ENDTODO
    ########################################

    return x

In [4]:
test_two_layer_perceptron.TestPreprocess.eval(preprocess_fn=preprocess)

test_preprocess (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_preprocess) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.026s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

# Dvouvrstvý perceptron

- Model bude optimalizovat křížovou entropii
  $$
  l_n = -\log\frac{\exp{s_{n,y_n}}}{\sum_{k=1}^{K}{\exp{s_{n,k}}}}
  $$
  kde skóre $\boldsymbol{s}_n$ bude výsledek dopředného průchodu dvouvrstvého perceptronu se sigmoidou $\sigma(\cdot)$ jako aktivační funkcí
  $$
  \begin{align*}
  \boldsymbol{r}_n &= \boldsymbol{x}_n \cdot \boldsymbol{w}^{(1)} + \boldsymbol{b}^{(1)} \\
  \boldsymbol{h}_n &= \sigma(\boldsymbol{r}_n) \\
  \boldsymbol{s}_n &= \boldsymbol{x}_n \cdot \boldsymbol{w}^{(2)} + \boldsymbol{b}^{(2)} \\
  \end{align*}
  $$
- Klasifikátor bude implementován jako třída [`ans.classification.TwoLayerPerceptron`](../ans/classification.py).
- Třída obsahuje parametry klasifikátoru jako atributy  
  | atribut   | značení                                                            | rozměr       |
  |-----------|--------------------------------------------------------------------|--------------|
  | `weight1` | $\boldsymbol{w}^{(1)} = \left[w_{d,k}^{(1)}\right]$                | $D \times H$ |
  | `bias1`   | $\boldsymbol{b}^{(1)} = \left[b_1^{(1)}, \ldots, b_H^{(1)}\right]$ | $H$          |
  | `weight2` | $\boldsymbol{w}^{(2)} = \left[w_{d,k}^{(2)}\right]$                | $H \times K$ |
  | `bias2`   | $\boldsymbol{b}^{(2)} = \left[b_1^{(2)}, \ldots, b_K^{(2)}\right]$ | $K$          |
- Třída bude mít shodné tři metody `__init__`, `train_step` a `val_step` jako `ans.classification.LinearSoftmaxModel`.
- Rozdílem bude pouze jejich vnitřní implementace.

### TODO: implementuje metodu [`ans.classification.TwoLayerPerceptron.__init__`](../ans/classification.py).

In [5]:
test_two_layer_perceptron.TestInit.eval()

test_init_params (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_init_params) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.012s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

### TODO: implementuje funkci [`ans.classification.TwoLayerPerceptron.train_step`](../ans/classification.py).

In [6]:
test_two_layer_perceptron.TestTrainStep.eval()

test_implementation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_implementation) ... ok
test_step (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_step) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.028s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

### TODO: implementuje funkci [`ans.classification.TwoLayerPerceptron.val_step`](../ans/classification.py).

In [7]:
test_two_layer_perceptron.TestValStep.eval()

test_implementation (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_implementation) ... ok
test_step (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_step) ... ok

----------------------------------------------------------------------
Ran 2 tests in 0.009s

OK


<unittest.runner.TextTestResult run=2 errors=0 failures=0>

# Trénování klasifikátoru

- Nejlepší model uložte jako
  ``` python
  model.save('../output/two_layer_perceptron_weights.pt')
  ```

### TODO: Natrénujte dvouvrstvý perceptron tak, aby dosáhl alespoň 45 % (bonusově 50 %) *validační* accuracy.

In [8]:
ans.utils.seed_everything(0)

batch_size = 256
learning_rate = 0.1
num_epochs = 40
hidden_size = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_dataset = torchvision.datasets.CIFAR10(root="../data", train=True, transform=preprocess, download=True)
val_dataset = torchvision.datasets.CIFAR10(root="../data", train=False, transform=preprocess, download=True)

sample_img, _ = train_dataset[0]
in_size = sample_img.numel()
out_size = len(train_dataset.classes)

train_loader = ans.data.BatchLoader(train_dataset, batch_size=batch_size, shuffle=True, device=device)
val_loader = ans.data.BatchLoader(val_dataset, batch_size=batch_size, shuffle=False, device=device)

model = ans.classification.TwoLayerPerceptron(in_size=in_size, hidden_size=hidden_size, out_size=out_size, weight_scale=0.1)

model.weight1 = model.weight1.to(device)
model.bias1 = model.bias1.to(device)
model.weight2 = model.weight2.to(device)
model.bias2 = model.bias2.to(device)

train_loss, train_acc = ans.classification.validate(model, train_loader)
val_loss, val_acc = ans.classification.validate(model, val_loader)
print(f"after init: train_loss={train_loss:.5f}, train_acc={train_acc:.3f}, val_loss={val_loss:.5f}, val_acc={val_acc:.3f}")

best_acc = 0.0

for epoch in range(num_epochs):
    ans.classification.train_epoch(model, train_loader, learning_rate=learning_rate)

    train_loss, train_acc = ans.classification.validate(model, train_loader)
    val_loss, val_acc = ans.classification.validate(model, val_loader)
    
    print(
        f"epoch {epoch + 1}: train_loss={train_loss:.5f}, train_acc={train_acc:.3f}, val_loss={val_loss:.5f}, val_acc={val_acc:.3f}"
    )

    if val_acc > best_acc:
        best_acc = val_acc
#batch = 250; alpha = 0.1; epoch = 40; hidden = 64 -> 0.4618
print(f"Nejlepší přesnost: {best_acc}")
model.save('../output/two_layer_perceptron_weights.pt')
print("Model byl úspěšně uložen!")

Files already downloaded and verified
Files already downloaded and verified
after init: train_loss=2.34179, train_acc=0.100, val_loss=2.34164, val_acc=0.101
epoch 1: train_loss=2.01909, train_acc=0.306, val_loss=2.01840, val_acc=0.311
epoch 2: train_loss=1.93302, train_acc=0.314, val_loss=1.93257, val_acc=0.319
epoch 3: train_loss=1.86093, train_acc=0.341, val_loss=1.86147, val_acc=0.340
epoch 4: train_loss=1.84380, train_acc=0.349, val_loss=1.84132, val_acc=0.350
epoch 5: train_loss=1.82894, train_acc=0.348, val_loss=1.82732, val_acc=0.347
epoch 6: train_loss=1.77243, train_acc=0.384, val_loss=1.77266, val_acc=0.383
epoch 7: train_loss=1.76486, train_acc=0.375, val_loss=1.76624, val_acc=0.373
epoch 8: train_loss=1.77701, train_acc=0.372, val_loss=1.77537, val_acc=0.371
epoch 9: train_loss=1.73876, train_acc=0.386, val_loss=1.74070, val_acc=0.386
epoch 10: train_loss=1.72113, train_acc=0.393, val_loss=1.72454, val_acc=0.386
epoch 11: train_loss=1.67826, train_acc=0.407, val_loss=1.6830

In [9]:
test_two_layer_perceptron.TestValAccuracy45.eval(preprocess_fn=preprocess)

test_val_acc (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_val_acc) ... ok

----------------------------------------------------------------------
Ran 1 test in 8.485s

OK


<unittest.runner.TextTestResult run=1 errors=0 failures=0>

In [ ]:
test_two_layer_perceptron.TestValAccuracy50.eval(preprocess_fn=preprocess)

test_val_acc (tests.ANSTestCase.eval.<locals>._TestCaseClass.test_val_acc) ... 